# Lesson 3: Redis Caching & Scalability Thinking
### VisionStream — From 100 Users to 100,000 Users

---

## Learning Objectives

By the end of this lesson, you will be able to:

1. **Explain** why caching is necessary in high-concurrency systems
2. **Implement** the Cache-Aside pattern with Redis and TTL
3. **Describe** horizontal scaling and load balancing concepts
4. **Write** production-quality Git commits using Conventional Commits
5. **Submit** a pull request with a clean, reviewable feature branch

## Section 1: Why We Need a Cache

### The Problem: High-Frequency Reads

The helmet's companion mobile app wants to show the user's **current location on a map**.
It polls the API every second: `GET /devices/{id}/location`.

With 10,000 helmets and a companion app each:
```
10,000 devices × 1 request/second = 10,000 database reads/second
```

A typical PostgreSQL instance handles ~1,000–5,000 reads/second before degrading.
We're 2–10× over capacity, and we haven't even counted the telemetry writes yet.

### The Solution: Redis Cache

Redis stores data **in memory** (RAM), not on disk. A Redis read takes ~0.1 ms.
A PostgreSQL read takes ~5–50 ms. Redis is **50–500× faster**.

| | PostgreSQL | Redis |
|---|---|---|
| Storage | Disk (durable) | RAM (volatile) |
| Read latency | 5–50 ms | 0.1–1 ms |
| Write throughput | ~10,000 writes/sec | ~100,000 writes/sec |
| TTL support | Manual cleanup | Built-in: SETEX or EXPIRE |
| Data loss on restart | Never | Yes (unless persistence enabled) |

### Key Insight: Not Everything Should Be Cached

Cache only data that is:
1. **Read frequently** (high read-to-write ratio)
2. **Acceptable to be slightly stale** (a 30-second-old GPS location is fine)
3. **Expensive to recompute** (complex SQL query results)

**Do NOT cache:**
- Financial transactions (must always be fresh)
- Authentication tokens (security risk)
- Large objects (Redis is expensive per GB)

## Section 2: Getting Redis Ready — Two Ways (Docker vs. Pure Python)

Now that you understand *why* caching matters, let's get a Redis server running
so you can use it in your code. This section covers **two** approaches.
Try the pure-Python approach first — it requires no Docker installation.

---

### Option A (Recommended): Start Redis with Python + subprocess

Redis is written in C, but you can download and run the Windows binary
directly from Python using the `subprocess` module. This works on any OS
where you can run `redis-server`.

If you don't have `redis-server` installed, install it:

- **Windows:** Download from https://github.com/redis-windows/redis-windows/releases
  (look for `Redis-x64-*.zip`), unzip, and add the folder to your `PATH`.
- **macOS:** `brew install redis`
- **Linux:** `sudo apt install redis-server`

Then start it from Python:



In [ ]:
# ── Start Redis server from Python ──
# This runs 'redis-server' as a background subprocess.
# The server listens on localhost:6379 by default.

import subprocess
import time

# Launch redis-server in the background
redis_process = subprocess.Popen(
    ["redis-server"],
    stdout=subprocess.DEVNULL,   # suppress logs (optional)
    stderr=subprocess.DEVNULL,
)

# Give it a second to finish starting up
time.sleep(1)

# Check if it's actually running
if redis_process.poll() is None:
    print("✅ Redis server is running on localhost:6379")
else:
    print("❌ Failed to start redis-server. Is it installed and on your PATH?")
    print("   Try Option B (Docker) below instead.")

When you're done, stop the server gracefully:


In [ ]:
# ── Stop the Redis server ──
# Send SIGTERM so Redis flushes data to disk before exiting
redis_process.terminate()
redis_process.wait(timeout=5)
print("🛑 Redis server stopped")

Once the server is running, verify you can connect to it using the `redis` Python package:


In [ ]:
# ── Verify connection with redis-py ──
# Make sure you have the package installed:
#   pip install redis

import redis

# Connect to the local Redis server
r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# Ping it — returns True if the server is reachable
if r.ping():
    print("✅ Successfully connected to Redis!")
    print(f"   Server info: {r.info('server')['redis_version']}")
else:
    print("❌ Could not connect to Redis. Is the server running?")

---

### Option B: Start Redis with Docker

If you have Docker installed and prefer not to install `redis-server` natively:

```bash
docker run --name visionstream-redis -p 6379:6379 -d redis:7-alpine
```

Verify it's running:

```bash
docker ps | grep visionstream-redis
```

Stop it later:

```bash
docker stop visionstream-redis
docker rm visionstream-redis
```

---

### Configure the REDIS_URL in Your App

Once Redis is running, make sure your `app/config.py` has the URL uncommented:

```python
REDIS_URL: str = "redis://localhost:6379/0"
```

And install the Python Redis client if you haven't already:

```bash
pip install redis
```

Now you're ready to code! The rest of this lesson assumes Redis is running
on `localhost:6379`.

---

## Section 3: The Cache-Aside Pattern

Cache-Aside (also called *lazy loading*) is the most common caching pattern.
The application code manages the cache manually.

### Cache-Aside Read Pattern

```
App → Redis GET "device:helmet-001:location"
         │
         ├── HIT  → Return cached value (fast: ~1ms)
         │
         └── MISS → Query PostgreSQL
                        │
                        └── Write result to Redis (cache it for next time)
                               └── Return value
```

### Cache-Aside Write Pattern (for telemetry upload)

```
POST /telemetry/upload received
         │
         Step 1: Write to PostgreSQL  (durable — must succeed first)
         │
         Step 2: Write to Redis with TTL=30s  (update the cache)
         │
         Return 201 Created
```

**Why write to PostgreSQL FIRST?**
If the Redis write fails (Redis is temporarily down), the data is still safely
stored in PostgreSQL. We never lose a sensor reading.

### TTL (Time-To-Live)

TTL defines how long a Redis key lives before it **automatically expires and is deleted**.

For device location:
- `TTL = 30 seconds`: If no new telemetry arrives in 30s, the key expires
- Result: Stale location data is never served longer than 30 seconds
- Result: No manual cleanup code needed — Redis handles expiry automatically

### Redis Key Naming Convention

```
Format:  "<resource>:<identifier>:<property>"
Example: "device:helmet-001-berkeley:location"

Other examples:
  "device:helmet-001:status"     → online/offline status
  "device:helmet-001:battery"    → battery level
  "session:user-42:token"        → auth token (different use case)
```

## Section 4: Illustrative Code — Redis Cache-Aside Pattern

Study the pattern below before implementing in `app/redis_client.py`
and `app/services/telemetry_service.py`.

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Study the pattern ──
import json

# Pattern: Setting a value in Redis with a TTL
def example_set_with_ttl(cache, key: str, value: dict, ttl_seconds: int) -> None:
    """
    Illustrates: store a dict in Redis as a JSON string with TTL.

    cache.setex(name, time, value) is equivalent to:
        SET key value
        EXPIRE key ttl_seconds
    """
    json_value = json.dumps(value)              # dict → JSON string
    # cache.setex(name=key, time=ttl_seconds, value=json_value)
    print(f"[ILLUSTRATIVE] Would SET {key} = {json_value} with TTL={ttl_seconds}s")


# Pattern: Getting a value from Redis (returns None on miss)
def example_get_from_cache(cache, key: str) -> dict | None:
    """
    Illustrates: read from Redis and deserialize.
    Returns None if key doesn't exist or has expired.
    """
    # raw = cache.get(key)          # Returns None on cache miss
    raw = None                      # placeholder
    if raw is None:
        print(f"[ILLUSTRATIVE] CACHE MISS for key: {key}")
        return None
    return json.loads(raw)          # JSON string → dict


# Pattern: Cache-Aside read (check cache first, fall back to DB)
def example_cache_aside_read(cache, db, device_id: str) -> dict | None:
    """
    Illustrates the full Cache-Aside read flow.
    """
    key = f"device:{device_id}:location"

    # Step 1: Check Redis
    location = example_get_from_cache(cache, key)
    if location is not None:
        print("[ILLUSTRATIVE] Cache HIT — served from Redis")
        return location

    # Step 2: Cache miss — query PostgreSQL
    print("[ILLUSTRATIVE] Cache MISS — querying PostgreSQL")
    # log = db.query(TelemetryLog).filter(...).order_by(...).first()
    log = None  # placeholder

    if log is None:
        return None

    # Step 3: Populate cache for next request
    location = {"gps_lat": 37.8719, "gps_lon": -122.2585, "timestamp": "..."}
    example_set_with_ttl(cache, key, location, ttl_seconds=30)
    return location


# Demonstrate the flow
print("First call (cold cache):")
example_cache_aside_read(None, None, "helmet-001")
print("\nSecond call (warm cache — would hit Redis in real implementation):")
print("[ILLUSTRATIVE] Subsequent calls return in ~0.1ms from Redis")

## Section 5: Scalability — From 100 to 100,000 Users

### The Single-Server Bottleneck

Right now, VisionStream runs on one server. What happens as usage grows?

```
100 devices:    API CPU: 2%   DB connections: 10    Redis: idle
1,000 devices:  API CPU: 20%  DB connections: 100   Redis: 5% memory
10,000 devices: API CPU: 95%  DB connections: 500   ← BOTTLENECK (DB maxes out)
100,000 devices: API CPU: ???  DB connections: ????  ← System crashes
```

### Solution: Horizontal Scaling

**Vertical scaling** = buy a bigger server (expensive, has limits)

**Horizontal scaling** = add more servers that share the load

```
                    ┌─────────────────────┐
Helmet devices  ──► │   Load Balancer     │ ← distributes incoming requests
                    │  (e.g., AWS ALB)    │   using Round Robin or Least Connections
                    └──────┬──────┬───────┘
                           │      │
                    ┌──────▼──┐ ┌─▼───────┐
                    │ API     │ │ API     │  ← Multiple API servers (stateless)
                    │ Server 1│ │ Server 2│
                    └──────┬──┘ └─┬───────┘
                           │      │
                    ┌──────▼──────▼──────┐
                    │   Shared Redis     │  ← Single cache (shared state)
                    └────────────────────┘
                    ┌────────────────────┐
                    │  PostgreSQL        │  ← Can add read replicas later
                    └────────────────────┘
```

### Why Redis Enables This Architecture

If device location was stored **in the API server's memory**, horizontal scaling would break:
- Request 1 (Server 1): Store location in Server 1's memory
- Request 2 (Server 2): Read location — **Server 2 doesn't know about it!**

By storing location in **shared Redis**, all servers see the same data. This is called
**stateless architecture** — a core principle of scalable systems.

## Section 6: Code Review Standards

Before you submit your PR for Lesson 3, review your work against these standards.

### Commit Message Quality

Your mentor will review your commit history. Use **Conventional Commits**:

```
# ✅ Good commits — clear, scoped, present tense
feat(cache): add Redis connection pool setup in redis_client.py
feat(telemetry): implement Cache-Aside write in telemetry_service.py
feat(api): update upload_telemetry router to use cache-aware service
refactor(device): simplify get_device_by_id query

# ❌ Bad commits — vague, past tense, too broad
updated stuff
fixed bug
WIP
changes
lesson 3 done
```

### Code Quality Checklist

| Check | Question |
|-------|----------|
| **Dead code** | Are there any commented-out blocks that should be deleted? |
| **Magic strings** | Is `"device:{device_id}:location"` defined as a constant? |
| **Error handling** | What happens if Redis is down? Does the app fall back gracefully? |
| **Type hints** | Do all functions have complete type annotations? |
| **Docstrings** | Does each public function have a docstring? |

### Feature Branch Workflow

```bash
# Create a feature branch from latest main
git checkout main
git pull origin main
git checkout -b feature/lesson3-redis-cache

# Make your changes, commit with Conventional Commits
git add app/redis_client.py app/services/telemetry_service.py
git commit -m "feat(cache): add Redis connection pool and set_device_location helper"

git add app/services/telemetry_service.py
git commit -m "feat(telemetry): implement Cache-Aside write pattern in create_telemetry_with_cache"

# Push and open PR
git push origin feature/lesson3-redis-cache
# Then open a Pull Request on GitHub
```

---

## 📌 YOUR TASK — File Map

### Step 1 — Redis Client (`app/redis_client.py`)

| Function | What to implement |
|----------|-------------------|
| `get_redis()` | Create Redis client using connection pool from `REDIS_URL` config |
| `set_device_location()` | `cache.setex(key, ttl, json.dumps(location))` |
| `get_device_location()` | `cache.get(key)` → `json.loads()` or None |
| `delete_device_location()` | `cache.delete(key)` |

### Step 2 — Telemetry Service (`app/services/telemetry_service.py`)

| Function | What to implement |
|----------|-------------------|
| `create_telemetry_with_cache()` | Call `create_telemetry_log()`, then `set_device_location()` |
| `get_latest_location()` | Cache-Aside read: Redis first, fallback to DB |

### Step 3 — Update the Telemetry Router (`app/routers/telemetry.py`)

- Add `cache: redis.Redis = Depends(get_redis)` parameter to `upload_telemetry()`
- Switch from `create_telemetry_log()` to `create_telemetry_with_cache()`

### Step 4 — Update the Devices Router (`app/routers/devices.py`)

- Add `cache: redis.Redis = Depends(get_redis)` to `get_device_location_endpoint()`
- Implement the Cache-Aside read: call `get_latest_location()` from telemetry_service

### Step 5 — Test with Redis Running

```bash
# Start Redis locally (or use Docker)
docker run -p 6379:6379 redis:7-alpine

# Or start all services:
docker-compose up db cache   # Start only DB and Redis (not the API)
uvicorn app.main:app --reload  # Start API separately (easier for development)

# Test the location endpoint:
# 1. Register a device
# 2. Upload telemetry
# 3. GET /devices/{device_id}/location
#    → First call: cache MISS (reads from DB, stores in Redis)
#    → Second call (within 30s): cache HIT (reads from Redis, ~1ms)
```

### Step 6 — Submit PR

```bash
git checkout -b feature/lesson3-redis-cache
# ... make commits ...
git push origin feature/lesson3-redis-cache
# Open PR on GitHub with description:
# Title: feat: implement Redis Cache-Aside pattern for device location
# Body: What changed, why, and how to test it
```

---

## ✅ Lesson 3 Self-Assessment Checklist

- [ ] Redis connection pool is set up in `app/redis_client.py`
- [ ] `set_device_location()` stores JSON with a 30-second TTL
- [ ] `get_device_location()` returns `None` on cache miss (not an exception)
- [ ] `create_telemetry_with_cache()` writes to PostgreSQL FIRST, then Redis
- [ ] `GET /devices/{id}/location` returns from Redis on second call (verify with logs)
- [ ] All new functions have type hints and docstrings
- [ ] Pull Request submitted on `feature/lesson3-redis-cache` branch
- [ ] PR has at least 3 commits with Conventional Commit messages
- [ ] PR description explains what changed and how to test
- [ ] No commented-out code left in committed files

---

**Bonus Challenge:** What happens to the location endpoint if Redis goes down?
Add error handling so a Redis `ConnectionError` causes a graceful fallback to PostgreSQL,
not a 500 Internal Server Error.

**When all boxes are checked → you are ready for Lesson 4.**